# Ablation 2: Clinical Conservative Augmentation

**Research Question**: Open: does clinical-conservative augmentation (rotation ±15°, h/v flip, brightness ±30%) improve performance.

In [ ]:
# Lock NumPy to prevent ultralytics from upgrading to 2.x
import numpy as np
import subprocess

default_numpy = np.__version__
print(f"Installing ultralytics...")
subprocess.run(["pip", "install", "-q", "ultralytics"], check=True)

print(f"Locking NumPy to {default_numpy}...")
subprocess.run(["pip", "install", "-q", "--force-reinstall", f"numpy=={default_numpy}"], check=True)

import numpy as np
print(f"OK NumPy locked: {np.__version__}")


In [ ]:
import sys
from pathlib import Path
import yaml
import shutil

# Environment
IN_KAGGLE = Path("/kaggle/input").exists()
BASE_INPUT_PATH = Path("/kaggle/input") if IN_KAGGLE else Path.cwd()
OUTPUT_PATH = Path("/kaggle/working") if IN_KAGGLE else Path("/tmp/ablation_2")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

# Code dataset
CODE_PATH = BASE_INPUT_PATH / "wound-seg-code-v1"
if CODE_PATH.exists():
    sys.path.insert(0, str(CODE_PATH))

print("OK Environment configured")


In [ ]:
# Copy pre-converted YOLO dataset from baseline-outputs
# baseline-outputs/data_wound_seg/ already has YOLO labels!

YOLO_SRC = BASE_INPUT_PATH / "baseline-outputs" / "data_wound_seg"
YOLO_DATASET_ROOT = OUTPUT_PATH / "data_wound_seg"

print("Copying YOLO dataset...")
if not YOLO_DATASET_ROOT.exists():
    shutil.copytree(YOLO_SRC, YOLO_DATASET_ROOT)
    print(f"OK Dataset copied to {YOLO_DATASET_ROOT}")

# Verify
train_imgs = len(list((YOLO_DATASET_ROOT / "train" / "images").glob("*.png")))
train_lbls = len(list((YOLO_DATASET_ROOT / "train" / "labels").glob("*.txt")))
test_imgs = len(list((YOLO_DATASET_ROOT / "test" / "images").glob("*.png")))
test_lbls = len(list((YOLO_DATASET_ROOT / "test" / "labels").glob("*.txt")))

print(f"OK Train: {train_imgs} images, {train_lbls} labels")
print(f"OK Test: {test_imgs} images, {test_lbls} labels")

if train_imgs == 0:
    raise ValueError("Dataset copy failed!")

# Create config
config_path = OUTPUT_PATH / "wound_dataset.yaml"
with open(config_path, 'w') as f:
    f.write(f"""path: {YOLO_DATASET_ROOT}
train: train
val: test

names:
  0: wound
""")

print(f"OK Config: {config_path}")


In [ ]:
from ultralytics import YOLO
import json
import time

RANDOM_SEEDS = [42, 123, 456]

for seed in RANDOM_SEEDS:
    print(f"\n{'='*80}")
    print(f"TRAINING SEED {seed}")
    print('='*80)

    # Create fresh model instance for each seed
    model = YOLO("yolov8n-seg.pt")

    start = time.time()
    results = model.train(
        data=str(config_path),
        epochs=100,
        batch=32,
        lr0=0.015,
        workers=8,
        amp=False,
        imgsz=512,
        patience=10,
        seed=seed,

        # CLINICAL AUGMENTATION (Only these enabled)
        degrees=15.0,        # ±15° rotation
        fliplr=0.5,          # Horizontal flip
        flipud=0.5,          # Vertical flip
        hsv_v=0.3,           # ±30% brightness

        # DISABLE ALL OTHER AUGMENTATIONS
        hsv_h=0.0, hsv_s=0.0,
        translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
        mosaic=0.0, mixup=0.0, copy_paste=0.0, erasing=0.0,
        auto_augment=None,
        close_mosaic=0,

        project=str(OUTPUT_PATH / "yolo_training"),
        name=f"seed_{seed}",
        exist_ok=True,
        verbose=False
    )

    duration = (time.time() - start) / 60
    print(f"OK Seed {seed} complete in {duration:.1f} min")

print("\nOK ALL TRAINING COMPLETE")


In [ ]:
# Save completion status
status = {
    "ablation": "2_clinical_augmentation",
    "status": "complete",
    "seeds": RANDOM_SEEDS
}

with open(OUTPUT_PATH / "completion_status.json", 'w') as f:
    json.dump(status, f, indent=2)

print("OK Ablation 2 complete!")
